## Formulario correo

In [1]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text

import numpy as np

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

engine_mysql = create_engine(
    f"mysql+pymysql://{user_envio}:{pwd_envio}@{server_envio}:{port_mysql}/{db_envio}"
)


In [2]:
filename='TARGET.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_target_desembolso = pd.read_csv(ruta_archivo,sep='|')

# df_target_desembolso.rename(columns={'DNI': 'dni_cliente'}, inplace=True)
# df_target_desembolso.rename(columns={'FECHA_DESEMBOLSOS': 'fecha_desembolso'}, inplace=True)
df_target_desembolso = df_target_desembolso[['DNI']].copy()
df_target_desembolso['target'] = 1

filename='ACUM_DESEM.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_fugas = pd.read_csv(ruta_archivo,sep='|')
df_fugas = df_fugas[['DNI','MONTO','ASESOR','CANALVENTA']].copy()

df_desembolso = df_target_desembolso[['DNI']].drop_duplicates().merge(
    df_fugas[['DNI']].drop_duplicates(),
    on=['DNI'],
    how='inner'
)

df_desembolso = df_target_desembolso.drop_duplicates().merge(
    df_fugas.drop_duplicates(),
    on=['DNI'],
    how='inner'
)
df_desembolso['target'] = df_desembolso['target'].fillna(0).astype(int)
df_desembolso['fugas'] = (df_desembolso['target'] == 0).astype(int)
df_desembolso.rename(columns={'DNI': 'dni_cliente'}, inplace=True)

df_desembolso['dni_cliente'] = (
    df_desembolso['dni_cliente']
    .astype(str)
    .str.replace(r'\D', '', regex=True)   
    .replace('', pd.NA)                     
    .str.zfill(8)                           
)



In [3]:
query = f"""
	SELECT * FROM Alice.prospectos_envio_alfin 
    where estado='procesado'
	and DATE(fecha_envio)>='2026-07-01'
"""
df_prospectos_envio_alfin = pd.read_sql(query, engine_mysql)

query = f"""
	SELECT * FROM Alice.prospectos_correos_alfin 
    where estado='ENVIADO'
	and DATE(fecha_envio)>='2026-07-01'
"""
df_prospectos_correos_alfin = pd.read_sql(query, engine_mysql)


In [4]:
df_prospectos_correos_alfin['dia_ref'] = pd.to_datetime(
    df_prospectos_correos_alfin['fecha_envio']
).dt.date

df_prospectos_envio_alfin['dia_ref'] = pd.to_datetime(
    df_prospectos_envio_alfin['fecha_envio']
).dt.date

df_prospectos_envio_alfin = df_prospectos_envio_alfin.drop_duplicates(subset=['dni_cliente','dia_ref'])
df_prospectos_correos_alfin = df_prospectos_correos_alfin.drop_duplicates(subset=['dni_cliente','dia_ref'])

df_seguimiento = df_prospectos_envio_alfin[['dni_cliente', 'dia_ref']].merge(
    df_prospectos_correos_alfin[['dni_cliente', 'dia_ref','celular']],
    on=['dni_cliente', 'dia_ref'],
    how='inner'
)

In [5]:
df_seguimiento_1 = df_seguimiento.merge(
    df_desembolso,
    on=['dni_cliente'],
    how='left'
)

df_seguimiento_1['target'] = df_seguimiento_1['target'].fillna(0).astype(int)
df_seguimiento_1['fugas'] = df_seguimiento_1['fugas'].fillna(0).astype(int)

In [6]:
filename='RetiroDefinitivo_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_def_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_Telefonos.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_telf = pd.read_csv(ruta_archivo,sep='|')
filename='retiro_correo_alfin.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_retiro_correo = pd.read_csv(ruta_archivo,sep=';')

filename='desembolso.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_des = pd.read_csv(ruta_archivo,sep=';')

df_def_blacklist = df_def_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_blacklist = df_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_telf= df_telf.rename(columns={'TELEFONO': 'celular'})
df_retiro_correo= df_retiro_correo.rename(columns={'DNI': 'dni_cliente'})


# Blacklists de DNI
df6 = df_des.copy()
df6["celular"] = None
df6 = df6[["dni_cliente", "celular"]]

# Blacklists de DNI
df1 = df_def_blacklist.copy()
df1["celular"] = None
df1 = df1[["dni_cliente", "celular"]]

df2 = df_blacklist.copy()
df2["celular"] = None
df2 = df2[["dni_cliente", "celular"]]

# Blacklist de teléfonos
df3 = df_telf.copy()
df3["dni_cliente"] = None
df3 = df3[["dni_cliente", "celular"]]

# Archivo con DNI y celular
df4 = df_retiro_correo[["dni_cliente", "celular"]].copy()



# Unir todo
df_retiros = pd.concat(
    [df1, df2, df3, df4,df6],
    ignore_index=True
)

dni_retiro = set(df_retiros['dni_cliente'].dropna())
cel_retiro = set(df_retiros['celular'].dropna())

df_seguimiento_1['retiro'] = (
    df_seguimiento_1['dni_cliente'].isin(dni_retiro) |
    df_seguimiento_1['celular'].isin(cel_retiro)
).astype(int)

C:\Users\DATA\AppData\Local\Temp\ipykernel_14680\3506000229.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_retiros = pd.concat(


### spark -- datos faltantes

In [7]:
ruta_archivo = os.path.join(ruta_csv, 'tmp_muestra_alfin.csv')
df_seguimiento_1['dni_cliente'].to_csv(ruta_archivo, index=False,sep=';')
# ruta_archivo = os.path.join(ruta_alfin, 'desembolso.csv')
# df_des.to_csv(ruta_archivo, index=False,sep=';')

In [8]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

In [ ]:
filename='Consulta_de_Campañas_202607_V2_SS_EXT_CAMBIO_db.csv'
df_validar_01=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
filename='Consulta_de_Campañas_202607_V2_SS_EXT_CAMBIO_db2.csv'
df_validar_02=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
df_validar=df_validar_01.unionByName(df_validar_02)

filename='tmp_muestra_alfin.csv'
df_tmp_dni=cargar_archivo_csv(spark,filename,';',True)

filename='retiro_alfin_acum.csv'
df_tmp_retiro=cargar_archivo_csv(spark,filename,';',True)

In [ ]:
# filename='Libro10.csv'
# aja=cargar_archivo_csv(spark,filename,';',True)
# aja = aja.withColumn(
#     "DNI",
#     F.right(
#         F.concat(F.lit("00000000"), F.col("DNI")),
#         F.lit(8)
#     )
# )
# overwrite_table_SQL(spark,aja,f'ALFIN_BORRAR_BORRAR_1',server_kishin,user_kishin,pwd_kishin,'DANTALION')

In [10]:

df_tmp_dni = df_tmp_dni.withColumn(
    "dni_cliente",
    F.right(
        F.concat(F.lit("00000000"), F.col("dni_cliente")),
        F.lit(8)
    )
)

df_validar = df_validar.withColumn(
    "dni_cliente",
    F.right(
        F.concat(F.lit("00000000"), F.col("DNI")),
        F.lit(8)
    )
).drop('DNI')

In [13]:
df_tmp_retiro = df_tmp_retiro.withColumn(
    "dni_cliente",
    F.right(
        F.concat(F.lit("00000000"), F.col("dni_cliente")),
        F.lit(8)
    )
).drop('DNI')

In [16]:
overwrite_table_SQL(spark,df_tmp_dni,f'tmp_muestra_cliente_alfin_borrar',server_kishin,user_kishin,pwd_kishin,'DANTALION')
# overwrite_table_SQL(spark,df_tmp_retiro,f'tmp_muestra_cliente_alfin_borrar_retiro_1',server_kishin,user_kishin,pwd_kishin,'DANTALION')


In [17]:
query = """
    select * from DANTALION.dbo.tmp_muestra_cliente_alfin_borrar 
    """
df_dni_ref=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)

df_validar=df_validar.join(df_dni_ref,['dni_cliente'],'inner')

query = """
    select a.NUMERO_DOCUMENTO as dni_cliente
    from DANTALION.dbo.Base_Maestra_ALFIN_BK_Vigente a
    inner join tmp_muestra_cliente_alfin_borrar b
    on a.NUMERO_DOCUMENTO=b.dni_cliente
    """
df_ref_base=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)

df_validar_pd = df_validar.toPandas()
df_ref_base_pd = df_ref_base.toPandas()

### completar base + columnas de consulta campaña

In [18]:
dni_en_base = set(df_ref_base_pd['dni_cliente'].dropna())

df_seguimiento_1['en_base'] = (
    df_seguimiento_1['dni_cliente'].isin(dni_en_base) 
).astype(int)

In [19]:
df = df_seguimiento_1.copy()

# Asegurar que dia_ref sea fecha
df['dia_ref'] = pd.to_datetime(df['dia_ref'], errors='coerce')

# Cantidad de envíos por DNI
df['q_envios'] = df.groupby('dni_cliente')['dni_cliente'].transform('size')

df_resumen = (
    df.sort_values('dia_ref')
      .drop_duplicates(subset='dni_cliente', keep='last')
      .copy()
)

import numpy as np

hoy = np.datetime64('today', 'D')

df_resumen['conteo_dia'] = np.busday_count(
    df_resumen['dia_ref'].values.astype('datetime64[D]'),
    hoy,
    weekmask='1111110'   # Lunes a sábado
)

# Crear env_1 hasta env_6
for i in range(1, 7):
    df_resumen[f'env_{i}'] = (df_resumen['q_envios'] == i).astype(int)

In [20]:
df_validar_pd_seleccion=df_validar_pd[['dni_cliente','COLOR_FINAL','USER_V3','FRESCURA','PROPENSION_DISTRIBUCION','OFERTA_MAX']]

In [21]:
df_resumen_1=df_resumen.merge(df_validar_pd_seleccion,on='dni_cliente',how='left')
df_resumen_1 = df_resumen_1.drop_duplicates(subset=['dni_cliente'])


In [22]:
ruta_archivo = os.path.join(ruta_csv, 'tmp_resumen1.csv')
df_resumen_1.to_csv(ruta_archivo, index=False,sep=';')

In [25]:
fechas = pd.to_datetime(['2026-07-18', '2026-07-20'])

df_resumen_1_2=df_resumen_1[
    (df_resumen_1['fugas'] == 0) &
    (df_resumen_1['retiro'] == 0) &
    (df_resumen_1['dia_ref'].isin(fechas)) &
    # (df_resumen_1['q_envios']==1) &
    (df_resumen_1['MONTO'].isna()) 
].copy()

In [ ]:
df_resumen_1_2.shape

(2235, 23)

In [56]:
df_resumen_1['PROPENSION_DISTRIBUCION'].unique()

array(['2', '4', '6', '5', '1', nan, '3'], dtype=object)

In [27]:
df_resumen_1_2.groupby('q_envios').size().reset_index(name='cantidad')

,q_envios,cantidad
0,2,358
1,3,1423
2,4,357
3,5,97


In [37]:
df_resumen_1.groupby(
    ['dia_ref', 'q_envios']
).size().reset_index(name='cantidad')

,dia_ref,q_envios,cantidad
0,2026-07-15,1,932
1,2026-07-15,2,458
2,2026-07-16,1,1708
3,2026-07-16,3,1


In [26]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)


In [27]:
df_resumen_1[df_resumen_1['MONTO'].isna()].head()

,dni_cliente,dia_ref,celular,target,MONTO,ASESOR,CANALVENTA,fugas,retiro,en_base,q_envios,conteo_dia,env_1,env_2,env_3,env_4,env_5,env_6,COLOR_FINAL,USER_V3,FRESCURA,PROPENSION_DISTRIBUCION,OFERTA_MAX
994,44355573,2026-07-09,910154097,0,NaN,NaN,NaN,0,0,0,1,10,1,0,0,0,0,0,NARANJA OSCURO,3. MES + PLD Peers,0,1,3000
995,44431542,2026-07-09,997580003,0,NaN,NaN,NaN,0,0,0,1,10,1,0,0,0,0,0,AMARILLO CLARO,2. sunedu & sunarp B,4,2,4200
996,16505328,2026-07-09,964869744,0,NaN,NaN,NaN,0,0,0,1,10,1,0,0,0,0,0,VERDE OSCURO,3. MES + PLD Peers,4,1,10600
997,15959521,2026-07-09,936093252,0,NaN,NaN,NaN,0,0,0,1,10,1,0,0,0,0,0,AMARILLO OSCURO,3. MES + PLD Peers,0,1,9800
998,15451499,2026-07-09,986015152,0,NaN,NaN,NaN,0,0,0,1,10,1,0,0,0,0,0,VERDE OSCURO,3. MES + PLD Peers,4,1,13400


In [58]:
df_resumen_1=df_resumen_1.rename(columns={'COLOR_FINAL':'color'})

In [29]:
df_resumen_1.shape

(730, 23)

In [59]:
df_prospectos_correos_alfin=df_prospectos_correos_alfin.drop(columns='color')


In [60]:
# df_prospectos_correos_alfin=df_prospectos_correos_alfin.drop(columns='color')
df_prospectos_envio_alfin=df_prospectos_envio_alfin.merge(df_resumen_1[['dni_cliente','color']],on='dni_cliente',how='inner')
df_prospectos_correos_alfin=df_prospectos_correos_alfin.merge(df_resumen_1[['dni_cliente','color']],on='dni_cliente',how='inner')

In [61]:
df_prospectos_envio_alfin = df_prospectos_envio_alfin.drop_duplicates(subset=['dni_cliente'])
df_prospectos_correos_alfin = df_prospectos_correos_alfin.drop_duplicates(subset=['dni_cliente'])


In [220]:
df_prospectos_envio_alfin=df_prospectos_envio_alfin.drop(columns='color')

In [64]:
# df_prospectos_envio_alfin=df_prospectos_envio_alfin.merge(df_resumen_1[['dni_cliente','COLOR_FINAL']],on='dni_cliente',how='inner')
# df_prospectos_correos_alfin=df_prospectos_correos_alfin.merge(df_resumen_1[['dni_cliente','COLOR_FINAL']],on='dni_cliente',how='inner')

df_prospectos_correos_alfin=df_prospectos_correos_alfin[['canal_campo', 'supervisor', 'ejecutivo_target', 'codigo_ejecutivo_id', 'cdv_alfin_banco', 'dni_cliente', 'nombre_cliente', 'color', 'monto_solicitado', 'celular', 'agencia_atencion', 'fecha_visita','hora_visita']] .copy()
df_prospectos_correos_alfin['tipo_carga']='MANUAL'

df_prospectos_envio_alfin=df_prospectos_envio_alfin[['dni_vendedor', 'operador', 'dni_cliente', 'nombre_cliente', 'telefono_cliente', 'agencia_tienda', 'fecha_visita', 'monto_solicitado', 'tipo_gestion']].copy()
df_prospectos_correos_alfin['fecha_visita']='2026-07-22'
df_prospectos_envio_alfin['fecha_visita']='2026-07-22'

df_prospectos_correos_alfin = df_prospectos_correos_alfin.drop_duplicates(subset='dni_cliente')
df_prospectos_envio_alfin = df_prospectos_envio_alfin.drop_duplicates(subset='dni_cliente')

display(df_prospectos_correos_alfin.head(2))
display(df_prospectos_envio_alfin.head(2))


,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,color,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita,tipo_carga
0,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,00000001,BOT,ROSA HONOR,21830475,ALFREDO ELEODORO REYES,AMARILLO OSCURO,7700.0,956559337,CHINCHA,2026-07-22,0 days 17:15:00,MANUAL
1,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,00000001,BOT,ROSA HONOR,22194568,MARIA IRENE RAMOS DE CRUZ,NaN,3100.0,999580241,TRUJILLO AMERICA,2026-07-22,0 days 17:15:00,MANUAL


,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion
0,00000001,TARGET,16767136,FRANCISCO TIMANA BECERRA,954391558,737870 - VILLA MARIA 2,2026-07-22,7700,Derivacion
1,00000001,TARGET,47587005,LUIS ARNOLD YUPANQUI GUTIERREZ,963742019,734265 - TRUJ CENTRO,2026-07-22,14000,Derivacion


In [63]:
df_prospectos_correos_alfin.shape

(2290, 14)

In [65]:
df_prospectos_envio_alfin = df_prospectos_envio_alfin.drop_duplicates(subset=['dni_cliente'])
df_prospectos_correos_alfin = df_prospectos_correos_alfin.drop_duplicates(subset=['dni_cliente'])


In [ ]:


df_prospectos_correos_alfin.to_sql(
    name="prospectos_correos_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

df_prospectos_envio_alfin.to_sql(
    name="prospectos_envio_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

C:\Users\DATA\AppData\Local\Temp\ipykernel_8480\3762344665.py:1: UserWarning: the 'timedelta' type is not supported, and will be written as integer values (ns frequency) to the database.
  df_prospectos_correos_alfin.to_sql(


2290

: 

In [164]:
query = f"""
		SELECT dni_cliente FROM Alice.prospectos_envio_alfin 
    where fecha_visita='2026-07-20'
"""
df_estan = pd.read_sql(query, engine_mysql)

In [165]:
df_resumen_1 = df_resumen_1[
    ~df_resumen_1['dni_cliente'].isin(df_estan['dni_cliente'])
].copy()
df_resumen_1.shape

(0, 23)